# 08 · Monitoring

**Role of this layer.** Every stage of every run (and every replay) appends one
JSON object to `runs/events.jsonl`. The log is append-only, so it doubles as an
audit trail and could be shipped to a log platform as is.

The Monitoring page reads two sources, and the functions below (the ones behind
the page) turn them into numbers:
- the **run records**, one per run in its final state: activity, human review,
  tokens and cost, latency and failures;
- the **event log**, for what can happen several times in a run: guardrail
  firings, capability executions and replays.

To make sure there is something to show, this notebook first runs the six example
workflows in mock mode (approving only 1 and 2, as `cli demo` does).

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.engine import run_request
from quantlab.mock_scenarios import SCENARIOS

for scenario in SCENARIOS:
    session = run_request(
        scenario.request, list(scenario.dataset_ids), "mock", approve=scenario.auto_approve
    )
    print(scenario.id, session.status)

chf_safe_haven_success completed
spread_frequency_mismatch_block blocked
arima_forecast_block blocked
prompt_injection_unknown_capability blocked


vix_full_sample_lookahead completed


buy_the_dip_tech completed


In [3]:
from quantlab.lineage import load_runs
from quantlab.monitoring import activity, monitored_runs, read_events

records = load_runs(settings.runs_dir)
events = read_events(settings.events_path)
counted = monitored_runs(records)
print(len(records), "run records,", len(counted), "counted by the page,", len(events), "events")
pd.Series(activity(records))

132 run records, 25 counted by the page, 1529 events


total              132.000000
completed           52.000000
blocked             48.000000
failed               0.000000
rejected             3.000000
completed_share      0.393939
blocked_share        0.363636
failed_share         0.000000
rejected_share       0.022727
dtype: float64

The page counts only the decided runs started on the Analysis page
(`monitored_runs`, printed above), and replays from every source: these headless runs
are not among them. To show what each function computes, the cells below apply them
to every run record.

In [4]:
from quantlab.monitoring import (
    grounded_narratives,
    planner_refusals,
    replay_results,
    review_outcomes,
)

pd.Series(
    {
        **review_outcomes(records),
        **replay_results(events),
        **grounded_narratives(records),
        **planner_refusals(records),
    }
)

approved             49.000000
rejected              3.000000
approval_rate         0.942308
edited                6.000000
edit_rate             0.122449
approval_median_s     0.037000
approval_p95_s       82.353000
replays               7.000000
matching              7.000000
replay_match_rate     1.000000
narratives           52.000000
grounded             52.000000
grounded_rate         1.000000
refused              28.000000
refused_share         0.212121
dtype: float64

Every guardrail and every capability, zeros included, most frequent first:

In [5]:
from quantlab.monitoring import capability_counts, guardrail_counts

pd.DataFrame(guardrail_counts(events))

,code,name,action,count
0,V10,MATERIAL_ASSUMPTION,APPROVE,244
1,V09,LOOK_AHEAD,APPROVE,38
2,P01,PLANNER_REFUSED,BLOCK,28
3,V08,FREQUENCY_MISMATCH,BLOCK,19
4,V03,INVALID_PARAM,BLOCK,16
5,E04,ALIGNMENT_LOSS,WARN,14
6,V02,UNKNOWN_CAPABILITY,BLOCK,9
7,R01,REQUEST_TOO_LONG,BLOCK,1
8,V01,INVALID_PLAN,BLOCK,0
9,V04,TYPE_MISMATCH,BLOCK,0


In [6]:
pd.DataFrame(capability_counts(events))

,capability,code,count
0,load_series,C01,64
1,flag_regime,C07,48
2,resample,C02,47
3,forward_return,C08,32
4,compare_groups,C09,32
5,hac_mean_difference,C10,32
6,compute_returns,C03,28
7,rolling_volatility,C06,25
8,spread,C05,6
9,difference,C04,0


Cost per run, from the tokens the API reported and the prices in
`config.LLM_PRICES_PER_MILLION_TOKENS`. Mock runs call no LLM and are left out, so
in mock mode the table has no runs and the cache saved nothing:

In [7]:
from quantlab.config import LLM_PRICES_PER_MILLION_TOKENS
from quantlab.monitoring import cache_savings, cost_table

display(pd.DataFrame(cost_table(records, LLM_PRICES_PER_MILLION_TOKENS)))
pd.Series(cache_savings(records, LLM_PRICES_PER_MILLION_TOKENS))

,stage,runs,input_mean,input_p95,output_mean,output_p95,cost_mean,cost_p95,cost_total
0,planner,24,3642.041667,5004.0,363.500000,908.20,0.000546,0.000898,0.013103
1,narrator,24,480.375000,1578.0,58.541667,174.35,0.000077,0.000231,0.001855
2,total,24,4122.416667,5958.0,422.041667,960.25,0.000623,0.001055,0.014958


planner_calls             24.000000
cache_hits                 4.000000
hit_rate                   0.166667
saved_input_tokens     17504.000000
saved_output_tokens     2985.000000
saved_cost                 0.003243
dtype: float64

Latency per stage (mean and 95th percentile, milliseconds), and the most recent
failures with the stage and the guardrail that stopped them:

In [8]:
from quantlab.monitoring import latency_table, recent_failures

pd.DataFrame(latency_table(records))

,stage,runs,mean_ms,p95_ms,recorded
0,Request check,0,NaN,NaN,False
1,Planner,131,892.162595,7240.2000,True
2,Validator,131,0.141145,0.3400,True
3,Executor,52,122.271731,205.3315,True
4,Narrator,52,561.525000,3015.2550,True


In [9]:
pd.DataFrame(recent_failures(records, limit=5))

,time,run_id,status,stage,code,message
0,2026-10-18T21:52:36.865Z,20261018-215236-97aed5,blocked,Validator,V02,The plan has blocking issues: V02:s2
1,2026-10-18T21:52:36.839Z,20261018-215236-3056be,blocked,Planner,P01,The plan has blocking issues: P01:plan
2,2026-10-18T21:52:36.811Z,20261018-215236-a58b9c,blocked,Validator,V08,The plan has blocking issues: V08:s4
3,2026-10-18T21:43:01.707Z,20261018-214301-6e81fc,blocked,Planner,P01,The plan has blocking issues: P01:plan
4,2026-10-18T21:31:04.945Z,20261018-213104-1acca3,blocked,Validator,V02,The plan has blocking issues: V02:s2
